# Projection of an Upscaled Spline
Let $f_{0}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f_{0}(x)=\sum_{k\in{\mathbb{Z}}}\,c_{0}[{k\bmod K_{0}}]\,\beta^{n_{0}}(x-\delta x_{0}-k)$ be the realization of a periodic random spline at nominal scale, with a specified period $K_{0},$ degree $n_{0},$ and delay $\delta_{0}.$ We display in thick gray a version $f_{\uparrow M}$ of this spline magnified by the positive integer magnification factor $M\in{\mathbb{N}}+1.$ Then, let $\tilde{f}_{\uparrow M}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto\tilde{f}_{\uparrow M}(x)=\sum_{k\in{\mathbb{Z}}}\,\tilde{c}_{\uparrow M}^{n}[{k\bmod K}]\,\beta^{n}(x-\delta x-k)$ be the spline of period $K=M\,K_{0},$ arbitrary degree $n,$ arbitrary delay $\delta x,$ and spline coefficients $\tilde{c}_{\uparrow M}^{n}$ chosen such that the mean-square criterion $J=\int_{0}^{K}\,\left(\tilde{f}_{\uparrow M}(x)-f_{0}(\frac{x}{M})\right)^{2}\,{\mathrm{d}}x$ is minimized. We display $\tilde{f}_{\uparrow M}$ in blue, with samples at the integers indicated by circles and stem lines, and knots shown as black dots. The boundaries of one period are highlighted in red.

Then, we print a numeric estimate of the integral (over one period) of the product between the projection $\tilde{f}_{\uparrow M}$ and the residue $\left(\tilde{f}_{\uparrow M}-f_{\uparrow M}\right).$ For optimal spline coefficients $\tilde{c}_{\uparrow M}^{n},$ the scalar product is expected to vanish. This is precisely what happens, up to numerical accuracy.

Finally, we print the same quantity but we avoid numeric estimates, relying instead on direct computations that are based on convolutions.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import scipy
import warnings

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_magnif = 5 # Maximal magnification factor

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline
f0 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 0)

# Plot
def update_plot (
    period0 = 6,
    degree0 = 0,
    delay0 = 0.0,
    magnif = 2,
    degree = 3,
    delay = 0.0
):
    global f0

    # Update of the spline
    if f0.period != period0:
        f0 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period0),
            degree = f0.degree
        )
    f0.degree = degree0
    f0.delay = delay0

    # Upscaling and projection
    gm = f0.upscaled_projected(magnification = magnif, degree = degree, delay = delay)

    # Dynamic range
    image = {f0.image(), gm.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plots
    subplot = plt.subplots()
    # Plot of the spline being magnified
    f0m = f0.upscaled(magnification = magnif)
    f0m.plot(
        subplot,
        plotpoints = 200 + 1,
        plotrange = plotrange,
        curve_fmt = "#e0e0e0",
        curve_lw = 7.0,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )

    # Plot of the projected spline
    gm.plot(subplot, plotpoints = 200 + 1, plotrange = plotrange)

    # Final display
    plt.show()

    # Numeric estimate of the scalar product
    def integrand (
        x
    ):
        return (gm.at(x) - f0.at(x / magnif)) * gm.at(x)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        djn = scipy.integrate.quad(
            integrand,
            0,
            magnif * period0,
            points = np.concatenate((np.multiply(magnif, f0.get_knots()), gm.get_knots())),
            limit = f0.period + gm.period + 1
        )
    display(Math(
        r"""
        \int_{{0}}^{{{0:}}}\,
        \tilde{{f}}_{{\uparrow{1:}}}(x)\,
        \left(\tilde{{f}}_{{\uparrow{1:}}}(x)-f_{{\uparrow{1:}}}(x))\right)\,
        {{\mathrm{{d}}}}x={2:.2E}
        """.format(gm.period, magnif, djn[0])
    ))

    # Convolution-based scalar product
    gmm = gm.mirrored()
    djc = (sk.PeriodicSpline1D.convolve(gmm, gm).at(0) -
        sk.PeriodicSpline1D.convolve(gmm, f0m).at(0))
    display(Math(
        r"""
        \left(\tilde{{f}}_{{\uparrow{0:}}}^{{\vee}}*\tilde{{f}}_{{\uparrow{0:}}}\right)(0)-
        \left(\tilde{{f}}_{{\uparrow{0:}}}^{{\vee}}*f_{{\uparrow{0:}}}\right)(0)={1:.2E}
        """.format(magnif, djc)
    ))

# Interaction
widgets.interactive(
    update_plot,
    period0 = (1, max_period),
    degree0 = (0, max_degree),
    delay0 = (-max_delay, max_delay),
    magnif = (1, max_magnif),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay)
)
